# Sensitive-vs-not — span classifier training (Colab)

Trains `microsoft/mdeberta-v3-base` as a marked-span MASK/KEEP classifier.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. On CPU this run takes
~4.7 hours (measured 2026-07-18, 147 steps at ~114 s/step); on a T4 it is minutes.

---

## Rules this notebook enforces (not just documents)

| Rule | Where |
|---|---|
| Only `llm_synthetic` data may leave local infra | Cell 4 calls `assert_upload_allowed(..., "colab")` and **stops** otherwise |
| The locked exam never comes here | Cell 4 rejects any `split="eval"` row |
| Never train on the exam | `assert_no_eval_in_train` inside the training script |
| `xlm-roberta-base` is forbidden | training script raises |

🔴 **Dev metrics from this notebook are NOT ship evidence.** They are for tuning only.
The only ship signal is the Task 17/19 eval on the locked `human_simulated` exam, run
**locally**. Do not quote a dev number as model quality.

⚠️ Data files are gitignored, so `git clone` gets the **code only**. You must upload
`train.jsonl` and `dev.jsonl` by hand in cell 3. That is deliberate — the repo is not a
channel for training data.

## 1. Confirm the GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the code and install

In [ ]:
REPO = "https://github.com/JeffTiong1031/Vanguard.git"
BRANCH = "ml-sensitive-vs-not"

!git clone --depth 1 --branch {BRANCH} {REPO} /content/Vanguard
%cd /content/Vanguard/ml
!pip install -q -e .
!pip install -q "transformers>=4.44" datasets accelerate sentencepiece

import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)
print("NOTE: the training script supports both transformers 4.x and 5.x APIs.")
print("      The 5.x path is verified locally; if 4.x errors here, report the traceback.")

## 3. Upload the training data

Upload **`train.jsonl`** and **`dev.jsonl`** (from `ml/data/train/` on your machine).

🔴 **Do not upload `exam.jsonl`.** The next cell rejects it, but do not rely on that —
the exam is the locked measuring instrument and stays on local infra.

In [ ]:
import pathlib
from google.colab import files

dest = pathlib.Path("/content/Vanguard/ml/data/train")
dest.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, blob in uploaded.items():
    (dest / name).write_bytes(blob)
    print("saved", dest / name, len(blob), "bytes")

## 4. Policy gate — this cell stops the run if the data should not be here

In [ ]:
from pathlib import Path
from sens.validate_jsonl import load_jsonl, validate_path
from sens.residency import assert_upload_allowed, counsel_gate_required

TRAIN = Path("data/train/train.jsonl")
DEV = Path("data/train/dev.jsonl")

for p in (TRAIN, DEV):
    if not p.exists():
        raise SystemExit(f"missing {p} — upload it in cell 3")
    errs = validate_path(p)
    if errs:
        raise SystemExit(f"{p} failed validation:\n" + "\n".join(errs[:10]))
    print("validated", p)

rows = load_jsonl(TRAIN) + load_jsonl(DEV)

# ADR 0023: real-provenance data must never reach Colab.
assert_upload_allowed(rows, target="colab")

# The exam is split="eval" and must never be uploaded here.
leaked = [e.id for e in rows if e.split == "eval"]
if leaked:
    raise SystemExit(f"eval-split rows present ({leaked[:5]}) — the exam does not belong on Colab.")

# ADR 0015 counsel STOP re-arms the moment real personal data appears.
if counsel_gate_required(rows):
    raise SystemExit("real-provenance data present — counsel gate re-armed, stop and escalate.")

provs = {e.provenance for e in rows}
print(f"OK  {len(rows)} rows, provenance={provs}, no eval rows, cleared for Colab.")

## 5. Train

`--mask-weight 1.0` is correct for the current corpus: MASK is the **majority** class
(216 of 388 train instances). The plan's class weighting exists for realistic data where
KEEP dominates — raise it only if dev shows the model collapsing toward KEEP.

In [ ]:
!python scripts/train_span_clf.py \
  --train data/train/train.jsonl \
  --dev   data/train/dev.jsonl \
  --out   artifacts/runs/colab_v1 \
  --epochs 3 --batch 16 --mask-weight 1.0

## 6. Read the result honestly

In [ ]:
import json
m = json.load(open("artifacts/runs/colab_v1/metrics.json"))
pr, rc = m["eval_mask_precision"], m["eval_mask_recall"]
print(json.dumps(m, indent=2))
print()
if rc <= 0.0:
    print("FAIL  MASK recall is 0 — the model collapsed to always-KEEP.")
    print("      ship_status() returns NOT_SHIPPED for this regardless of precision.")
    print("      Try more epochs, or --mask-weight > 1.0.")
elif pr >= 0.999 and rc >= 0.999:
    print("SUSPICIOUS  perfect dev scores on 51 MASK spans usually means leakage,")
    print("            not skill. Check the train/dev split before believing it.")
else:
    print(f"dev MASK precision={pr:.3f} recall={rc:.3f}")
print()
print("Dev has ~51 MASK spans -> 95% CI is roughly +/-8pp. This is a trend, not a measurement.")
print("Dev and train share a source (LLM draft + human audit), so this is same-distribution")
print("performance. It is NOT evidence of field accuracy and NOT a ship signal.")

## 7. Download the checkpoint

Unzip into `ml/artifacts/runs/colab_v1/` locally. Task 17 evaluates it against the locked
exam **on local infra** — that run, not this one, answers whether the model is any good.

In [ ]:
!cd artifacts/runs && zip -qr /content/colab_v1.zip colab_v1
from google.colab import files
files.download("/content/colab_v1.zip")